# ServeNow: SLA Breach Prediction

XGBoost binary classification for the ServeNow Workforce Operating System.

---

## Section 1. Business and Model Objective

### Business problem

ServeNow currently depends on managers to notice when internal work is falling behind.
Detection happens by manual review, which means a task is usually identified as at risk
only after the SLA has already been missed or is about to be missed. The operational
consequence is reactive firefighting rather than planned intervention.

### Prediction objective

Given the information available about a task at the moment it is created and assigned,
estimate the probability that the task will breach its assigned SLA.

### Target definition

`sla_breached`

- `1` the task breached its assigned SLA
- `0` the task was completed within its assigned SLA

### Prediction timing

The prediction is made at the assignment and monitoring stage, before the task is completed.

**The model predicts risk before task completion. It must not use post-completion information.**

Any variable that only becomes known once the task is finished, for example actual completion
time or a recorded breach reason, would inflate measured performance and would be unavailable
at the moment the prediction is actually needed. Section 6 audits this explicitly.

### Why SLA breach prediction matters

An SLA breach carries a contractual cost, a customer relationship cost, and an internal cost
in the form of escalation handling. Most breaches are preceded by observable operating
conditions such as assignee overload, unresolved dependencies, or a task sitting too long in
the queue. Those conditions are visible before the deadline passes, which makes the problem
learnable from data.

### How the prediction connects to management intervention

The model output is a ranked risk signal, not a decision. It routes attention:

```
Task created -> Task assigned -> Model scores breach probability
    Low risk    -> normal workflow
    Medium risk -> manager monitoring
    High risk   -> recommend intervention
```

The manager decides what to do. Section 19 sets out this workflow in full, including the
constraint that the model must never be used to automatically penalise an employee.

## Section 2. Environment Setup

Install and pin the required libraries, then set the reproducibility and display
configuration used by every later cell.

In [ ]:
# Colab ships most of these. Install quietly and only what the notebook actually uses.
!pip install -q xgboost shap joblib

In [ ]:
import os
import io
import json
import zipfile
import warnings
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

import xgboost as xgb
from xgboost import XGBClassifier

import shap
import joblib

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Display configuration
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# Plot configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"

warnings.filterwarnings("ignore", category=FutureWarning)

print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost     :", xgb.__version__)
print("shap        :", shap.__version__)
print("joblib      :", joblib.__version__)
print("random seed :", RANDOM_SEED)

## Section 3. Data Loading

The notebook looks for the CSV on the Colab filesystem first. If it is not present, it falls
back to the Colab manual upload widget so the notebook still runs end to end.

In [ ]:
CANDIDATE_PATHS = [
    "/content/servenow_sla_breach_mock_data.csv",
    "/content/servenow_sla_breach_mock_data_5000.csv",
    "servenow_sla_breach_mock_data.csv",
    "servenow_sla_breach_mock_data_5000.csv",
]


def load_dataset(paths=CANDIDATE_PATHS):
    # Try known locations, then fall back to the Colab upload widget.
    for path in paths:
        if os.path.exists(path):
            print("Loaded from:", path)
            return pd.read_csv(path)

    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(
            "CSV not found. Place servenow_sla_breach_mock_data.csv in the working directory."
        )

    print("File not found on disk. Please upload the CSV.")
    uploaded = files.upload()
    filename = next(iter(uploaded))
    print("Loaded from upload:", filename)
    return pd.read_csv(io.BytesIO(uploaded[filename]))


df = load_dataset()
TARGET = "sla_breached"

In [ ]:
print("Shape:", df.shape)
print()
print("First 5 rows")
display(df.head())
print("Last 5 rows")
display(df.tail())

In [ ]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
    "example": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
})
overview.index.name = "column"
display(overview)

### Interpretation

The dataset holds one row per task, with a mix of task attributes, customer attributes,
assignee attributes, and operating state captured at the prediction point. Two columns carry
a small share of missing values, which Section 4 examines before any treatment is applied.
The target is present and binary, so the problem is set up as supervised binary classification.

## Section 4. Data Quality Audit

Every check below reports a finding without changing the data. Treatment decisions are made
in a separate cell so that each change is visible and justified.

In [ ]:
def audit_dataframe(frame, target):
    # Build a per column quality table. No data is modified here.
    numeric_cols = frame.select_dtypes(include=[np.number]).columns

    rows = []
    for col in frame.columns:
        series = frame[col]
        is_num = col in numeric_cols
        rows.append({
            "column": col,
            "dtype": str(series.dtype),
            "missing": int(series.isna().sum()),
            "missing_pct": round(series.isna().mean() * 100, 2),
            "n_unique": int(series.nunique()),
            "negative_values": int((series < 0).sum()) if is_num else 0,
            "min": series.min() if is_num else None,
            "max": series.max() if is_num else None,
        })

    return pd.DataFrame(rows).set_index("column")


quality_table = audit_dataframe(df, TARGET)
display(quality_table)

In [ ]:
# Structural and logical checks
checks = []

checks.append(("Duplicate rows", int(df.duplicated().sum())))
checks.append(("Duplicate task_id", int(df["task_id"].duplicated().sum())))
checks.append(("Columns with any missing", int((df.isna().sum() > 0).sum())))
checks.append(("Numeric columns with negatives",
               int((df.select_dtypes(include=[np.number]) < 0).any().sum())))
checks.append(("Infinite values",
               int(np.isinf(df.select_dtypes(include=[np.number]).to_numpy(dtype=float)).sum())))
checks.append(("remaining_sla_hours > sla_hours",
               int((df["remaining_sla_hours"] > df["sla_hours"] + 1e-6).sum())))
checks.append(("sla_hours <= 0", int((df["sla_hours"] <= 0).sum())))
checks.append(("current_workload_ratio <= 0", int((df["current_workload_ratio"] <= 0).sum())))
checks.append(("task_complexity outside 1 to 5",
               int((~df["task_complexity"].between(1, 5)).sum())))
checks.append(("employee_historical_sla_rate outside 0 to 1",
               int((~df["employee_historical_sla_rate"].between(0, 1)).sum())))
checks.append(("Target values outside {0, 1}",
               int((~df[TARGET].isin([0, 1])).sum())))

checks_df = pd.DataFrame(checks, columns=["check", "violations"])
checks_df["status"] = np.where(checks_df["violations"] == 0, "PASS", "REVIEW")
display(checks_df)

In [ ]:
# Categorical consistency: look for casing or whitespace variants of the same label
categorical_cols = df.select_dtypes(include=["object", "string"]).columns.drop("task_id")

for col in categorical_cols:
    values = df[col].dropna().unique()
    normalised = pd.Series([str(v).strip().lower() for v in values])
    has_variants = normalised.duplicated().any()
    has_whitespace = any(str(v) != str(v).strip() for v in values)
    print(f"{col:<28} categories={len(values):<3} "
          f"case_or_duplicate_variants={has_variants}  padded_whitespace={has_whitespace}")
    print("   ", sorted(map(str, values)))
    print()

In [ ]:
# Class distribution
class_counts = df[TARGET].value_counts().sort_index()
class_share = df[TARGET].value_counts(normalize=True).sort_index()

class_summary = pd.DataFrame({
    "count": class_counts,
    "share": (class_share * 100).round(2),
})
class_summary.index = ["0 (within SLA)", "1 (breached)"]
display(class_summary)

print("Positive class rate:", round(df[TARGET].mean(), 4))

In [ ]:
# Treatment decisions. Each change is explicit and reversible.
TREATMENTS = []

# Finding 1: two columns contain missing values below 5 percent.
# Why it matters: an unhandled NaN either crashes the pipeline or is silently imputed
#   with a value the business cannot interpret.
# Treatment: retain the NaN and let XGBoost learn a default split direction, which is its
#   native missing value handling. Add explicit binary indicators so the fact of missingness
#   remains available to the model and auditable by a reviewer.
missing_cols = df.columns[df.isna().any()].tolist()
TREATMENTS.append({
    "finding": f"Missing values in {missing_cols}",
    "why_it_matters": "Silent imputation hides a real operational gap in the source system.",
    "treatment": "Retain NaN for native XGBoost handling and add binary missingness indicators.",
    "applied_in": "Section 7 feature engineering",
})

# Finding 2: identifier columns are present.
# Why it matters: task_id is unique per row and carries no signal. employee_id is a
#   high cardinality personal identifier and raises a governance concern.
# Treatment: exclude both from the feature matrix. See Section 6.
TREATMENTS.append({
    "finding": "Identifier columns task_id and employee_id present",
    "why_it_matters": "task_id is row unique noise. employee_id encodes a person, not a condition.",
    "treatment": "Exclude both from the feature matrix.",
    "applied_in": "Section 6 feature selection",
})

display(pd.DataFrame(TREATMENTS))
print("Rows removed:", 0)
print("Values overwritten:", 0)

### Interpretation

The dataset passes the structural checks: no duplicate rows, no duplicate `task_id`, no
negative or infinite values, and no logically impossible records such as remaining SLA time
exceeding the total SLA window. Categorical labels are internally consistent, so no
normalisation is required.

Two quality items need a decision rather than a fix. The missing values sit below the five
percent threshold in both affected columns and reflect a real gap in the source system rather
than a corruption, so they are retained and flagged instead of imputed. The identifier columns
are dropped for the reasons recorded in the treatment table. No rows were removed and no
values were overwritten.

The class distribution is imbalanced but workable. A classifier that predicts the majority
class for every task would already score the majority share as accuracy, which is why
Section 13 reports precision, recall, and PR-AUC rather than accuracy alone.

## Section 5. Exploratory Data Analysis

The analysis below is limited to what informs modelling decisions. Correlations describe
association in this dataset and do not establish causation.

### A. Target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
order = [0, 1]
sns.countplot(x=df[TARGET], order=order, ax=ax, color="#4C72B0")

total = len(df)
for patch, cls in zip(ax.patches, order):
    height = patch.get_height()
    ax.annotate(f"{height:,} ({height / total * 100:.1f}%)",
                (patch.get_x() + patch.get_width() / 2, height),
                ha="center", va="bottom", fontsize=10)

ax.set_title("SLA Breach Target Distribution")
ax.set_xlabel("sla_breached (0 = within SLA, 1 = breached)")
ax.set_ylabel("Number of tasks")
ax.set_xticklabels(["0 Within SLA", "1 Breached"])
ax.set_ylim(0, total * 0.85)
plt.tight_layout()
plt.show()

### Interpretation

Breached tasks are the minority class. The imbalance is moderate rather than extreme, so the
positive class has enough support for reliable estimation without resampling. The practical
consequence is that accuracy is a misleading headline metric here, and that the decision
threshold has to be chosen deliberately rather than left at the 0.50 default.

### B. Numeric feature distributions

In [ ]:
EDA_NUMERIC = [
    "current_workload_ratio",
    "task_complexity",
    "estimated_task_hours",
    "sla_hours",
    "remaining_sla_hours",
    "dependency_delay_hours",
    "employee_historical_sla_rate",
]

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
axes = axes.ravel()

for ax, col in zip(axes, EDA_NUMERIC):
    sns.histplot(df[col].dropna(), bins=40, ax=ax, color="#4C72B0")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Tasks")

for ax in axes[len(EDA_NUMERIC):]:
    ax.set_visible(False)

fig.suptitle("Distribution of Representative Numeric Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

display(df[EDA_NUMERIC].describe().T)

### Interpretation

The effort and duration variables are right skewed, which is expected for operational work
where most tasks are small and a long tail of large tasks exists. That skew is not a problem
for a tree based model, since XGBoost splits on order rather than on magnitude, so no log
transform or scaling is applied.

Workload ratio is centred close to normal capacity with a meaningful share of tasks assigned
above it, meaning overload is a real and measurable condition in the data rather than a rare
edge case. Historical SLA rate is concentrated at the high end, so the informative variation
sits in the lower tail: the signal comes from the few assignees who consistently underperform.

### C. SLA breach rate by categorical segment

In [ ]:
CATEGORICAL_SEGMENTS = ["task_type", "task_priority", "customer_tier", "employee_department"]

ORDERED_LEVELS = {
    "task_priority": ["Low", "Medium", "High", "Critical"],
    "customer_tier": ["Standard", "Growth", "Enterprise", "Strategic"],
}


def breach_rate_by(frame, column, target=TARGET):
    grouped = frame.groupby(column)[target].agg(tasks="size", breach_rate="mean")
    if column in ORDERED_LEVELS:
        grouped = grouped.reindex(ORDERED_LEVELS[column])
    else:
        grouped = grouped.sort_values("breach_rate", ascending=False)
    return grouped


overall_rate = df[TARGET].mean()
fig, axes = plt.subplots(2, 2, figsize=(15, 9))

for ax, col in zip(axes.ravel(), CATEGORICAL_SEGMENTS):
    stats = breach_rate_by(df, col)
    sns.barplot(x=stats["breach_rate"] * 100, y=stats.index, ax=ax, color="#4C72B0")
    ax.axvline(overall_rate * 100, color="#C44E52", linestyle="--", linewidth=1.2,
               label=f"Overall {overall_rate * 100:.1f}%")
    ax.set_title(f"SLA breach rate by {col}")
    ax.set_xlabel("Breach rate (%)")
    ax.set_ylabel("")
    ax.legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()

for col in CATEGORICAL_SEGMENTS:
    print(f"--- {col} ---")
    display(breach_rate_by(df, col).assign(breach_rate=lambda d: (d["breach_rate"] * 100).round(2)))

### Interpretation

Breach rate varies materially across every segment, which confirms the categorical variables
carry usable signal rather than noise. Priority and customer tier show the clearest gradient,
which is consistent with how the SLA policy is written: higher tiers and higher priorities
receive tighter windows, so the same amount of work has less room before it is late.

That last point matters for how the results are read. A higher breach rate for Strategic
customers reflects a stricter contractual clock, not worse service delivery to those accounts.
Reporting the raw segment rate to stakeholders without that context would invite the wrong
conclusion.

### D. Relationship between operating conditions and breach rate

In [ ]:
RELATIONSHIP_FEATURES = [
    "current_workload_ratio",
    "dependency_delay_hours",
    "remaining_sla_hours",
]


def binned_breach_rate(frame, column, bins=8, target=TARGET):
    # Quantile bins keep group sizes comparable on skewed variables.
    binned = pd.qcut(frame[column], q=bins, duplicates="drop")
    return frame.groupby(binned, observed=True)[target].agg(tasks="size", breach_rate="mean")


fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

for ax, col in zip(axes, RELATIONSHIP_FEATURES):
    stats = binned_breach_rate(df, col)
    positions = np.arange(len(stats))
    ax.plot(positions, stats["breach_rate"] * 100, marker="o", color="#4C72B0")
    ax.axhline(overall_rate * 100, color="#C44E52", linestyle="--", linewidth=1.2)
    ax.set_xticks(positions)
    ax.set_xticklabels([f"{iv.left:,.1f}" for iv in stats.index], rotation=45, ha="right")
    ax.set_title(f"Breach rate across {col} bins")
    ax.set_xlabel(f"{col} (quantile bin lower edge)")
    ax.set_ylabel("Breach rate (%)")

plt.tight_layout()
plt.show()

for col in RELATIONSHIP_FEATURES:
    print(f"--- {col} ---")
    display(binned_breach_rate(df, col).assign(
        breach_rate=lambda d: (d["breach_rate"] * 100).round(2)))

### Interpretation

Breach rate rises monotonically with workload ratio and with dependency delay, and falls as
remaining SLA time increases. All three gradients run in the direction operational experience
would predict, which is a useful sanity check on the data before any model is fitted.

The workload curve steepens above normal capacity rather than rising linearly, which suggests
the cost of overload accelerates once an assignee passes their capacity point. This is an
association observed in the data. It does not establish that reducing workload would cause a
proportional reduction in breaches, since assignment is not random and harder tasks may be
routed to already busy specialists.

### E. Correlation analysis

In [ ]:
numeric_for_corr = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_for_corr].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=False, linewidths=0.4, cbar_kws={"label": "Pearson correlation"}, ax=ax)
ax.set_title("Correlation Matrix, Numeric Variables")
plt.tight_layout()
plt.show()

target_corr = (corr[TARGET].drop(TARGET).sort_values(ascending=False).to_frame("corr_with_target"))
display(target_corr)

### Interpretation

No individual feature shows a correlation with the target strong enough to solve the problem
on its own. The strongest linear associations sit in the moderate range, which is the expected
shape for an operational risk problem where outcomes arise from combinations of conditions
rather than from a single dominant driver. This is also the case for a gradient boosted model,
which can represent the interactions that a linear correlation cannot detect.

Some feature pairs are strongly correlated with each other, for example effort estimates and
SLA duration, because the SLA policy is derived from the effort estimate. Tree ensembles
tolerate this collinearity for predictive purposes, but it means importance can be split
across correlated features, so Section 15 interprets feature groups rather than reading a
single ranking position as definitive.

## Section 6. Data Leakage Audit

This section is mandatory. A leaked feature produces strong offline metrics and a model that
cannot run in production, because the leaked value does not exist at prediction time.

In [ ]:
FORBIDDEN_FEATURES = [
    "actual_completion_time",
    "actual_resolution_time",
    "post_completion_status",
    "breach_reason",
    "manager_intervention_after_breach",
    "escalation_after_breach",
]

present_forbidden = [c for c in FORBIDDEN_FEATURES if c in df.columns]

print("Explicit post completion columns checked:", len(FORBIDDEN_FEATURES))
print("Found in dataset:", present_forbidden if present_forbidden else "none")

if present_forbidden:
    df = df.drop(columns=present_forbidden)
    print("Removed:", present_forbidden)

In [ ]:
# Screen for features that could indirectly encode the outcome.
# A near perfect association with the target is the signature of leakage.
numeric_features_only = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]

suspicion = pd.DataFrame({
    "abs_corr_with_target": df[numeric_features_only].corrwith(df[TARGET]).abs()
}).sort_values("abs_corr_with_target", ascending=False)

LEAKAGE_CORR_THRESHOLD = 0.90
suspicion["flag"] = np.where(
    suspicion["abs_corr_with_target"] >= LEAKAGE_CORR_THRESHOLD, "INVESTIGATE", "ok"
)
display(suspicion)

flagged = suspicion.index[suspicion["flag"] == "INVESTIGATE"].tolist()
print("Features flagged for investigation:", flagged if flagged else "none")

# Single feature separability check: can any one feature alone almost reproduce the target
single_feature_auc = {
    col: max(roc_auc_score(df[TARGET], df[col].fillna(df[col].median())),
             1 - roc_auc_score(df[TARGET], df[col].fillna(df[col].median())))
    for col in numeric_features_only
}
single_auc = pd.Series(single_feature_auc).sort_values(ascending=False).to_frame("univariate_auc")
single_auc["flag"] = np.where(single_auc["univariate_auc"] >= 0.95, "INVESTIGATE", "ok")
display(single_auc.head(10))

In [ ]:
# Final feature selection
IDENTIFIER_COLUMNS = ["task_id", "employee_id"]

RAW_CATEGORICAL_FEATURES = [
    "task_type",
    "task_priority",
    "customer_tier",
    "employee_department",
]

RAW_NUMERIC_FEATURES = [
    "employee_experience_years",
    "employee_historical_sla_rate",
    "current_open_tasks",
    "current_workload_ratio",
    "task_complexity",
    "estimated_task_hours",
    "sla_hours",
    "remaining_sla_hours",
    "dependency_count",
    "dependency_delay_hours",
    "reassignment_count",
    "similar_task_avg_hours",
    "employee_avg_completion_hours",
    "task_queue_age_hours",
    "customer_escalation_history",
    "cross_department_required",
    "peak_workload_flag",
]

expected = set(IDENTIFIER_COLUMNS + RAW_CATEGORICAL_FEATURES + RAW_NUMERIC_FEATURES + [TARGET])
unaccounted = set(df.columns) - expected
assert not unaccounted, f"Unaccounted columns: {unaccounted}"

print("Identifiers excluded :", IDENTIFIER_COLUMNS)
print("Categorical features :", len(RAW_CATEGORICAL_FEATURES))
print("Numeric features     :", len(RAW_NUMERIC_FEATURES))
print("Target               :", TARGET)

### Interpretation and feature selection rationale

None of the six explicitly prohibited post completion fields exist in the dataset, so nothing
had to be removed on that basis. The indirect screen also comes back clean: no feature reaches
the correlation threshold that would signal a disguised copy of the target, and no single
feature can reproduce the outcome on its own. Both screens are necessary, because a leaked
feature does not have to share the target's name to share its information.

Three features deserve a specific note. `employee_historical_sla_rate`,
`employee_avg_completion_hours`, and `similar_task_avg_hours` are aggregates built from earlier
tasks. In this synthetic dataset they are generated as point in time snapshots, so they are
safe. On real ServeNow data they must be recomputed strictly as of each task's creation
timestamp. Computing them over the full history, including tasks that closed later, is
look ahead leakage and is the failure mode most likely to pass unnoticed, because the resulting
column looks entirely ordinary.

Two columns are excluded from the feature matrix. `task_id` is unique per row and carries no
generalisable signal. `employee_id` is excluded for two reasons: it is a high cardinality
personal identifier that would let the model attach risk to a named individual rather than to
an operating condition, which conflicts with the governance constraint in Section 19, and
keeping it would require grouped cross validation to avoid identity leaking across splits.
Employee behaviour still reaches the model through the tenure, historical SLA rate, and average
completion time features, which describe conditions rather than identity.

## Section 7. Feature Engineering

Every feature below is a row wise transformation of values already known at prediction time.
None of them reference the target, and none of them require fitting on the data, so applying
the function to each split independently produces identical results and introduces no leakage.

In [ ]:
def safe_divide(numerator, denominator):
    # Return NaN rather than inf where the denominator is zero or negative.
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    return np.where(denominator > 0, numerator / np.where(denominator > 0, denominator, 1.0), np.nan)


def engineer_features(frame):
    # Pure row wise transformation. Safe to apply per split.
    out = frame.copy()

    out["estimated_vs_sla_ratio"] = safe_divide(out["estimated_task_hours"], out["sla_hours"])
    out["workload_pressure_score"] = np.clip(out["current_workload_ratio"] - 1.0, 0.0, None)
    out["dependency_pressure_score"] = out["dependency_count"] * np.log1p(out["dependency_delay_hours"])
    out["employee_speed_ratio"] = safe_divide(out["employee_avg_completion_hours"], out["sla_hours"])
    out["sla_buffer_ratio"] = safe_divide(out["remaining_sla_hours"], out["sla_hours"])
    out["queue_pressure"] = safe_divide(out["task_queue_age_hours"], out["sla_hours"])

    # Missingness indicators, the treatment recorded in Section 4
    out["employee_historical_sla_rate_missing"] = out["employee_historical_sla_rate"].isna().astype(int)
    out["similar_task_avg_hours_missing"] = out["similar_task_avg_hours"].isna().astype(int)

    return out


ENGINEERED_FEATURES = [
    "estimated_vs_sla_ratio",
    "workload_pressure_score",
    "dependency_pressure_score",
    "employee_speed_ratio",
    "sla_buffer_ratio",
    "queue_pressure",
    "employee_historical_sla_rate_missing",
    "similar_task_avg_hours_missing",
]

NUMERIC_FEATURES = RAW_NUMERIC_FEATURES + ENGINEERED_FEATURES
CATEGORICAL_FEATURES = RAW_CATEGORICAL_FEATURES
MODEL_FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

df_fe = engineer_features(df)

print("Engineered features:", len(ENGINEERED_FEATURES))
print("Total model input columns before encoding:", len(MODEL_FEATURES))
display(df_fe[ENGINEERED_FEATURES].describe().T)

assert np.isfinite(
    df_fe[ENGINEERED_FEATURES].to_numpy(dtype=float)[
        ~np.isnan(df_fe[ENGINEERED_FEATURES].to_numpy(dtype=float))]
).all(), "Infinite value produced by feature engineering"
print("No infinite values produced.")

### Feature definitions

| Feature | Formula | What it represents |
|---|---|---|
| `estimated_vs_sla_ratio` | `estimated_task_hours / sla_hours` | How much of the SLA window the estimated effort consumes. A value near or above 1 means the task needs nearly the whole window even under ideal conditions. |
| `workload_pressure_score` | `max(current_workload_ratio - 1, 0)` | Load above normal capacity only. Working at 80 percent and at 100 percent are both unpressured states, so both map to zero and the feature isolates genuine overload. |
| `dependency_pressure_score` | `dependency_count * log1p(dependency_delay_hours)` | Blocking exposure. The count alone ignores severity and the delay alone ignores breadth. The log dampens the long tail so one extreme delay does not dominate. |
| `employee_speed_ratio` | `employee_avg_completion_hours / sla_hours` | The assignee's typical pace measured against this specific deadline. It expresses fit between person and task rather than absolute speed. |
| `sla_buffer_ratio` | `remaining_sla_hours / sla_hours` | Share of the SLA window still unspent. Scale free, so a 2 hour and a 200 hour SLA become directly comparable. |
| `queue_pressure` | `task_queue_age_hours / sla_hours` | Share of the window already consumed while waiting. The complement of the buffer ratio in intent, but sourced from the queue side. |
| `employee_historical_sla_rate_missing` | `isna` indicator | Records that the assignee's SLA history was unavailable, which is itself an operational fact. |
| `similar_task_avg_hours_missing` | `isna` indicator | Records that no comparable historical cohort existed for this task. |

The ratio features exist because absolute hours are not comparable across tasks. Twelve
remaining hours is comfortable against a 200 hour SLA and close to failure against a 14 hour
SLA. Converting to a share of the window gives the model a quantity that means the same thing
across the whole task population, which is also what makes the resulting SHAP explanations
readable by a manager.

`safe_divide` returns NaN rather than infinity where a denominator is not strictly positive.
NaN is a value XGBoost handles natively, whereas an infinity propagates through the pipeline
and produces silent failures downstream.

## Section 8. Train, Validation and Test Split

The split happens before any fitted transformation. The validation set drives early stopping,
hyperparameter selection, and threshold selection. The test set is scored exactly once, at the
end, and is never used to make a modelling decision.

In [ ]:
X_all = df_fe[MODEL_FEATURES].copy()
y_all = df_fe[TARGET].copy()
meta_all = df_fe[["task_id", "employee_id"]].copy()

# 70 / 15 / 15, stratified on the target
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X_all, y_all,
    test_size=0.30,
    stratify=y_all,
    random_state=RANDOM_SEED,
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_holdout, y_holdout,
    test_size=0.50,
    stratify=y_holdout,
    random_state=RANDOM_SEED,
)

meta_test = meta_all.loc[X_test.index].copy()

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_validation), len(X_test)],
    "share": [len(X_train) / len(X_all), len(X_validation) / len(X_all), len(X_test) / len(X_all)],
    "positive_rate": [y_train.mean(), y_validation.mean(), y_test.mean()],
    "positives": [int(y_train.sum()), int(y_validation.sum()), int(y_test.sum())],
}, index=["train", "validation", "test"])
split_summary["share"] = (split_summary["share"] * 100).round(2)
display(split_summary)

assert len(set(X_train.index) & set(X_validation.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_validation.index) & set(X_test.index)) == 0
print("No row overlap between splits.")

In [ ]:
# Diagnostic: the same assignee appears in more than one split.
# This is acceptable here only because employee_id is excluded from the feature matrix.
train_employees = set(meta_all.loc[X_train.index, "employee_id"])
test_employees = set(meta_all.loc[X_test.index, "employee_id"])
overlap = train_employees & test_employees

print("Distinct employees in train:", len(train_employees))
print("Distinct employees in test :", len(test_employees))
print("Employees present in both  :", len(overlap))

### Interpretation

Stratification holds the positive rate close to constant across all three splits, so validation
and test estimates are measured against the same class balance the model was trained on.

The employee overlap diagnostic reports a real property of the data rather than a defect. Most
assignees appear in every split, which would inflate results if the model could memorise
individual identity. It cannot, because `employee_id` was excluded in Section 6 and the
assignee reaches the model only through condition based features such as tenure and historical
SLA rate. If a future version of this pipeline reintroduces `employee_id` as a feature, the
random split must be replaced with a grouped split keyed on that column, otherwise the reported
performance will be optimistic.

## Section 9. Preprocessing

The preprocessor is fitted on the training split only and then applied unchanged to validation
and test. Fitting it on the full dataset first would let information from the held out rows
influence the encoding.

In [ ]:
def build_onehot_encoder():
    # The sparse argument was renamed in scikit-learn 1.2.
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", build_onehot_encoder(), CATEGORICAL_FEATURES),
        ("numeric", "passthrough", NUMERIC_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

preprocessor.fit(X_train)
FEATURE_NAMES = list(preprocessor.get_feature_names_out())


def transform_to_frame(fitted_preprocessor, frame, feature_names):
    matrix = fitted_preprocessor.transform(frame)
    return pd.DataFrame(matrix, columns=feature_names, index=frame.index).astype(float)


X_train_processed = transform_to_frame(preprocessor, X_train, FEATURE_NAMES)
X_validation_processed = transform_to_frame(preprocessor, X_validation, FEATURE_NAMES)
X_test_processed = transform_to_frame(preprocessor, X_test, FEATURE_NAMES)

print("Encoded feature count:", len(FEATURE_NAMES))
print("Train     :", X_train_processed.shape)
print("Validation:", X_validation_processed.shape)
print("Test      :", X_test_processed.shape)
print()
print("Missing values retained for native XGBoost handling:")
print(X_train_processed.isna().sum()[lambda s: s > 0])

### Interpretation

Categorical variables are one hot encoded with `handle_unknown="ignore"`, so a category that
appears in production but never appeared in training produces an all zero block instead of an
error. Numeric variables pass through untransformed. Scaling and centring are deliberately
omitted: XGBoost splits on rank order, so a monotonic rescaling cannot change the tree
structure and would only make the SHAP output harder to read in business units.

Missing values are not imputed. XGBoost learns a default split direction for absent values
during training, which means the model can treat a missing SLA history as its own condition
rather than as an invented average. The explicit indicator columns added in Section 7 keep that
decision visible to a reviewer.

## Section 10. Baseline Model

A baseline establishes the score any real model has to beat. Without one, a headline metric
has no reference point.

In [ ]:
baseline = DummyClassifier(strategy="prior", random_state=RANDOM_SEED)
baseline.fit(X_train_processed, y_train)

baseline_probability = baseline.predict_proba(X_test_processed)[:, 1]
baseline_prediction = baseline.predict(X_test_processed)

baseline_metrics = {
    "accuracy": accuracy_score(y_test, baseline_prediction),
    "precision": precision_score(y_test, baseline_prediction, zero_division=0),
    "recall": recall_score(y_test, baseline_prediction, zero_division=0),
    "f1": f1_score(y_test, baseline_prediction, zero_division=0),
    "roc_auc": roc_auc_score(y_test, baseline_probability),
    "pr_auc": average_precision_score(y_test, baseline_probability),
}

display(pd.Series(baseline_metrics, name="baseline (prior)").to_frame().round(4))
print("Test set positive rate:", round(y_test.mean(), 4))

### Interpretation

The prior baseline predicts the majority class for every task. It reaches an accuracy equal to
the majority share while recalling none of the breaches, which is exactly the failure mode the
project exists to avoid: a model with a respectable accuracy number and zero operational value.

Its ROC-AUC of 0.50 and its PR-AUC equal to the positive class rate are the two reference
points that matter. Any lift the XGBoost model reports in Section 13 is measured against these
numbers, not against zero.

## Section 11. XGBoost Model

The starting configuration uses moderate values rather than extremes. `scale_pos_weight` is
computed from the training split to counteract the class imbalance without resampling.

In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
SCALE_POS_WEIGHT = negative_count / positive_count

BASE_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "scale_pos_weight": SCALE_POS_WEIGHT,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "tree_method": "hist",
}

print("Train class balance:", {"negative": negative_count, "positive": positive_count})
print("scale_pos_weight   :", round(SCALE_POS_WEIGHT, 4))

In [ ]:
def fit_with_early_stopping(params, X_tr, y_tr, X_va, y_va, early_stopping_rounds=50):
    # XGBoost moved early_stopping_rounds from fit() to the constructor in version 1.6.
    # Try the modern signature first, then fall back.
    try:
        model = XGBClassifier(**params, early_stopping_rounds=early_stopping_rounds)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return model
    except (TypeError, ValueError):
        pass

    model = XGBClassifier(**params)
    try:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                  early_stopping_rounds=early_stopping_rounds, verbose=False)
    except TypeError:
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return model


def best_round_count(model, fallback):
    # Number of boosting rounds to keep, given early stopping.
    best_iteration = getattr(model, "best_iteration", None)
    if best_iteration is None:
        return fallback
    return int(best_iteration) + 1


baseline_xgb = fit_with_early_stopping(
    BASE_PARAMS, X_train_processed, y_train, X_validation_processed, y_validation
)

baseline_xgb_rounds = best_round_count(baseline_xgb, BASE_PARAMS["n_estimators"])
baseline_xgb_val_probability = baseline_xgb.predict_proba(X_validation_processed)[:, 1]

print("Boosting rounds used:", baseline_xgb_rounds, "of", BASE_PARAMS["n_estimators"])
print("Validation ROC-AUC  :", round(roc_auc_score(y_validation, baseline_xgb_val_probability), 4))
print("Validation PR-AUC   :", round(average_precision_score(y_validation, baseline_xgb_val_probability), 4))

### Interpretation

Early stopping halts training when validation log loss stops improving, which sets the number
of boosting rounds from the data instead of from a guess. The gap between the rounds actually
used and the 300 round ceiling indicates whether the configured capacity was appropriate: a
model stopping far short of the ceiling has enough capacity, while one running to the ceiling
may benefit from more rounds or a lower learning rate.

Both validation scores sit well above the baseline from Section 10, so the features carry real
signal. Section 12 checks whether a different configuration does better before anything is
committed.

## Section 12. Model Selection and Tuning

A staged search rather than an exhaustive grid. Stage one tunes the parameters that control
model capacity. Stage two tunes the sampling parameters around the stage one winner. Every
candidate is trained on the training split and ranked on the validation split, so the test set
plays no part in selection.

In [ ]:
SELECTION_METRIC = "pr_auc"  # appropriate for an imbalanced positive class


def evaluate_candidate(params, early_stopping_rounds=50):
    model = fit_with_early_stopping(
        params, X_train_processed, y_train, X_validation_processed, y_validation,
        early_stopping_rounds=early_stopping_rounds,
    )
    probability = model.predict_proba(X_validation_processed)[:, 1]
    return {
        "rounds_used": best_round_count(model, params["n_estimators"]),
        "roc_auc": roc_auc_score(y_validation, probability),
        "pr_auc": average_precision_score(y_validation, probability),
    }


def run_search(grid, fixed_params):
    keys = list(grid.keys())
    results = []
    for combination in product(*(grid[k] for k in keys)):
        candidate = dict(fixed_params)
        candidate.update(dict(zip(keys, combination)))
        scores = evaluate_candidate(candidate)
        results.append({**dict(zip(keys, combination)), **scores})
    frame = pd.DataFrame(results).sort_values(SELECTION_METRIC, ascending=False)
    return frame.reset_index(drop=True)


stage_one_grid = {
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.03, 0.05, 0.10],
    "min_child_weight": [1, 3, 5],
}

stage_one_results = run_search(stage_one_grid, BASE_PARAMS)
print("Stage 1 candidates evaluated:", len(stage_one_results))
display(stage_one_results.head(10).round(4))

In [ ]:
stage_one_best = stage_one_results.iloc[0]

stage_two_fixed = dict(BASE_PARAMS)
stage_two_fixed.update({
    "max_depth": int(stage_one_best["max_depth"]),
    "learning_rate": float(stage_one_best["learning_rate"]),
    "min_child_weight": int(stage_one_best["min_child_weight"]),
})

stage_two_grid = {
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
}

stage_two_results = run_search(stage_two_grid, stage_two_fixed)
print("Stage 2 candidates evaluated:", len(stage_two_results))
display(stage_two_results.head(10).round(4))

In [ ]:
stage_two_best = stage_two_results.iloc[0]

BEST_PARAMS = dict(stage_two_fixed)
BEST_PARAMS.update({
    "subsample": float(stage_two_best["subsample"]),
    "colsample_bytree": float(stage_two_best["colsample_bytree"]),
})
BEST_N_ESTIMATORS = int(stage_two_best["rounds_used"])
BEST_PARAMS["n_estimators"] = BEST_N_ESTIMATORS

# Refit the final model without early stopping using the round count discovered above.
# This removes any dependency on a best_iteration attribute, so the saved artifact and the
# in memory model produce identical probabilities in the Section 21 reload test.
final_model = XGBClassifier(**BEST_PARAMS)
final_model.fit(X_train_processed, y_train, verbose=False)

tuned_val_probability = final_model.predict_proba(X_validation_processed)[:, 1]

comparison = pd.DataFrame({
    "starting configuration": {
        "roc_auc": roc_auc_score(y_validation, baseline_xgb_val_probability),
        "pr_auc": average_precision_score(y_validation, baseline_xgb_val_probability),
    },
    "tuned configuration": {
        "roc_auc": roc_auc_score(y_validation, tuned_val_probability),
        "pr_auc": average_precision_score(y_validation, tuned_val_probability),
    },
}).round(4)

print("Selected parameters")
for key in ["max_depth", "learning_rate", "min_child_weight", "subsample",
            "colsample_bytree", "n_estimators"]:
    print(f"  {key:<18}: {BEST_PARAMS[key]}")
print()
print("Validation comparison")
display(comparison)

### Interpretation

The staged search evaluates a small number of meaningful configurations instead of a large grid
that would mostly re-test near identical models. Ranking uses PR-AUC because the positive class
is the minority and the business cares about how well breaches are ranked, not about how well
the model separates the abundant negative class.

The improvement over the starting configuration is typically modest, which is the expected
result when the starting values were already reasonable. That is worth stating plainly to
stakeholders: most of the performance in this model comes from the features, not from
hyperparameter search. The final model is refit at the discovered round count without early
stopping so that the saved artifact and the in memory model are provably identical, which
Section 21 verifies.

## Section 13. Classification Metrics

All numbers below are computed on the test set, which has not influenced any decision up to
this point.

In [ ]:
def evaluate_at_threshold(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "f1": f1_score(y_true, prediction, zero_division=0),
        "flagged": int(prediction.sum()),
        "flagged_pct": float(prediction.mean() * 100),
    }


test_probability = final_model.predict_proba(X_test_processed)[:, 1]

test_roc_auc = roc_auc_score(y_test, test_probability)
test_pr_auc = average_precision_score(y_test, test_probability)

default_metrics = evaluate_at_threshold(y_test, test_probability, 0.50)

headline = pd.DataFrame({
    "baseline (prior)": {
        "accuracy": baseline_metrics["accuracy"],
        "precision": baseline_metrics["precision"],
        "recall": baseline_metrics["recall"],
        "f1": baseline_metrics["f1"],
        "roc_auc": baseline_metrics["roc_auc"],
        "pr_auc": baseline_metrics["pr_auc"],
    },
    "xgboost (threshold 0.50)": {
        "accuracy": default_metrics["accuracy"],
        "precision": default_metrics["precision"],
        "recall": default_metrics["recall"],
        "f1": default_metrics["f1"],
        "roc_auc": test_roc_auc,
        "pr_auc": test_pr_auc,
    },
}).round(4)

display(headline)
print("Test positive rate (PR-AUC floor):", round(y_test.mean(), 4))

In [ ]:
confusion = confusion_matrix(y_test, (test_probability >= 0.50).astype(int))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion, annot=True, fmt=",d", cmap="Blues", cbar=False,
            xticklabels=["Predicted within SLA", "Predicted breach"],
            yticklabels=["Actual within SLA", "Actual breach"], ax=ax)
ax.set_title("Confusion Matrix, Test Set, Threshold 0.50")
plt.tight_layout()
plt.show()

true_negative, false_positive, false_negative, true_positive = confusion.ravel()
print(f"True negatives  : {true_negative:,}")
print(f"False positives : {false_positive:,}  (task flagged, completed on time)")
print(f"False negatives : {false_negative:,}  (breach missed by the model)")
print(f"True positives  : {true_positive:,}")
print()
print(classification_report(y_test, (test_probability >= 0.50).astype(int),
                            target_names=["within SLA", "breached"], digits=4))

### Interpretation

The model separates the two classes substantially better than the prior baseline on both
ranking metrics. PR-AUC is the more informative of the two here, because it is measured against
the positive class rate rather than against 0.50, so the lift it reports is lift on the class
the business actually cares about.

### Why recall matters, and why it cannot be maximised alone

A false negative is a breach the model failed to flag. The manager receives no signal, no
intervention happens, and the SLA is missed with its full contractual and relationship cost.
A false positive is a task flagged that would have completed on time. The cost is a manager
spending review time on work that did not need it.

These costs are not symmetric, which argues for favouring recall. But the argument has a limit.
A model that flags every task achieves perfect recall and provides no information, because a
list that contains everything ranks nothing. The review capacity it consumes is real while the
prioritisation it provides is zero, and managers stop trusting the signal, which removes the
value of the correct flags as well.

The usable operating point sits between those extremes and depends on how much review capacity
ServeNow actually has. Section 14 makes that trade explicit rather than leaving it at the 0.50
default.

## Section 15 preview note: threshold first

Threshold selection comes before interpretability because the chosen threshold determines which
tasks are flagged, and the individual explanation in Section 16 is only meaningful for a task
the system would actually surface.

## Section 14. Threshold Analysis

The default 0.50 cut point assumes equal cost for both error types and a calibrated
probability. Neither assumption is safe here, so the threshold is treated as a business
parameter chosen on validation data and then reported on test.

In [ ]:
CANDIDATE_THRESHOLDS = [0.30, 0.40, 0.50, 0.60, 0.70]

validation_threshold_table = pd.DataFrame([
    evaluate_at_threshold(y_validation, tuned_val_probability, t) for t in CANDIDATE_THRESHOLDS
])
test_threshold_table = pd.DataFrame([
    evaluate_at_threshold(y_test, test_probability, t) for t in CANDIDATE_THRESHOLDS
])

print("Validation set, used for selection")
display(validation_threshold_table.round(4))

print("Test set, reported after selection")
display(test_threshold_table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(validation_threshold_table["threshold"], validation_threshold_table["precision"],
        marker="o", label="Precision")
ax.plot(validation_threshold_table["threshold"], validation_threshold_table["recall"],
        marker="o", label="Recall")
ax.plot(validation_threshold_table["threshold"], validation_threshold_table["f1"],
        marker="o", label="F1")
ax.set_title("Precision, Recall and F1 Across Decision Thresholds, Validation Set")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Select on validation, never on test.
SELECTED_THRESHOLD = float(
    validation_threshold_table.loc[validation_threshold_table["f1"].idxmax(), "threshold"]
)

selected_test_metrics = evaluate_at_threshold(y_test, test_probability, SELECTED_THRESHOLD)

print("Selected threshold:", SELECTED_THRESHOLD)
print("Selection basis   : highest F1 on the validation set")
print()
print("Test set performance at the selected threshold")
display(pd.Series(selected_test_metrics, name="value").to_frame().round(4))

review_load = selected_test_metrics["flagged"] / len(y_test)
print(f"Share of tasks routed for review: {review_load * 100:.1f}%")
print(f"Of those flagged, the share that truly breach: "
      f"{selected_test_metrics['precision'] * 100:.1f}%")
print(f"Of all real breaches, the share caught: "
      f"{selected_test_metrics['recall'] * 100:.1f}%")

### Interpretation and recommendation

Lowering the threshold raises recall and lowers precision, and each step also raises the number
of tasks routed for manager review. The `flagged` column translates the statistical trade into
the operational one, which is the version a manager can act on: it states how much review work
each threshold creates.

The threshold above was selected by maximising F1 on validation, which weights precision and
recall equally. That is a reasonable default in the absence of costed inputs, and it is not a
claim about the right business answer. A lower threshold is justified if ServeNow judges a
missed breach to be more expensive than a wasted review, which is the common case; a higher one
is justified if review capacity is the binding constraint.

The final threshold should be calibrated against real operational costs: the average cost of a
breach by customer tier, and the average cost of a manager review. With those two figures the
threshold becomes an expected cost minimisation rather than a statistical convention, and it
should be revisited whenever staffing or SLA terms change.

## Section 15. Model Interpretability

SHAP values decompose each individual prediction into per feature contributions that sum to the
model output. This makes the model auditable at the level of a single task, which is the
requirement for a system that asks a manager to act.

In [ ]:
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test_processed)

# Binary XGBClassifier returns a single matrix. Normalise the shape defensively.
if isinstance(shap_values, list):
    shap_values = shap_values[1]
shap_values = np.asarray(shap_values)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 1]

print("SHAP matrix shape:", shap_values.shape)
print("Test matrix shape:", X_test_processed.shape)

In [ ]:
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=FEATURE_NAMES)
global_importance = mean_abs_shap.sort_values(ascending=False)
TOP_FEATURES = global_importance.head(10)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=TOP_FEATURES.values, y=TOP_FEATURES.index, color="#4C72B0", ax=ax)
ax.set_title("Top 10 Features by Mean Absolute SHAP Value")
ax.set_xlabel("Mean absolute SHAP value (average impact on model output)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

display(TOP_FEATURES.to_frame("mean_abs_shap").round(5))

In [ ]:
shap.summary_plot(shap_values, X_test_processed, feature_names=FEATURE_NAMES,
                  max_display=15, show=False)
plt.title("SHAP Summary, Direction and Magnitude of Feature Effects", fontsize=12,
          fontweight="bold")
plt.tight_layout()
plt.show()

### Interpretation

The bar chart ranks features by average influence on the model output. The summary plot adds
direction: each point is one task, horizontal position is that feature's contribution to that
task's prediction, and colour encodes whether the underlying feature value was high or low. A
feature whose high values sit consistently on the right increases predicted breach risk as it
rises.

Reading the top ranked features in business terms:

- **SLA pressure features** such as `sla_buffer_ratio`, `queue_pressure`, and
  `estimated_vs_sla_ratio` describe how much of the deadline is already spent or committed.
  Low remaining buffer pushes predictions upward, which matches the operational reality that a
  task with little time left has no room to absorb any further delay.
- **Workload features** such as `current_workload_ratio` and `workload_pressure_score` capture
  assignee capacity. Load above the capacity point pushes risk up, and the effect strengthens
  rather than staying flat, which is why the engineered version isolates the overload region.
- **Dependency features** such as `dependency_pressure_score` and `dependency_delay_hours`
  represent blocking exposure. Work that cannot start is a distinct risk channel from work that
  is simply large, and the model separates the two.
- **Assignee history** such as `employee_historical_sla_rate` acts in the protective direction:
  a strong past record reduces predicted risk and partially offsets moderate load.

Two cautions apply. SHAP explains what the model learned, not what causes a breach in the
world. A feature ranking high means the model relies on it, which could reflect a genuine
mechanism or a proxy for something unobserved. And because several features are correlated by
construction, for example effort estimate and SLA duration, importance can be split across a
group. Read the feature families above rather than treating one rank position as decisive.

## Section 16. Individual Prediction Example

A global ranking tells a manager which factors matter on average. It does not tell them why
this specific task was flagged. The function below produces a single task explanation in a form
a non technical reviewer can act on.

In [ ]:
RISK_BANDS = [
    (0.60, "High"),
    (0.30, "Medium"),
    (0.00, "Low"),
]

# Business readable labels for the encoded feature names
FEATURE_LABELS = {
    "sla_buffer_ratio": "Low remaining SLA buffer",
    "queue_pressure": "Time already spent waiting in queue",
    "estimated_vs_sla_ratio": "Estimated effort large relative to the SLA window",
    "current_workload_ratio": "Assignee workload versus capacity",
    "workload_pressure_score": "Assignee overloaded above capacity",
    "dependency_pressure_score": "Blocking dependency exposure",
    "dependency_delay_hours": "Delay already caused by dependencies",
    "dependency_count": "Number of blocking dependencies",
    "employee_historical_sla_rate": "Assignee historical SLA record",
    "employee_speed_ratio": "Assignee typical pace versus this deadline",
    "employee_avg_completion_hours": "Assignee average completion time",
    "employee_experience_years": "Assignee tenure",
    "remaining_sla_hours": "Remaining SLA hours",
    "sla_hours": "Total SLA window",
    "estimated_task_hours": "Estimated effort",
    "task_complexity": "Task complexity",
    "task_queue_age_hours": "Queue waiting time",
    "reassignment_count": "Times the task changed owner",
    "customer_escalation_history": "Customer escalation history",
    "cross_department_required": "Requires another department",
    "peak_workload_flag": "Falls in a known peak period",
    "current_open_tasks": "Assignee open task count",
    "similar_task_avg_hours": "Historical time for similar tasks",
}


def risk_category(probability):
    for cutoff, label in RISK_BANDS:
        if probability >= cutoff:
            return label
    return "Low"


def readable_label(feature_name):
    if feature_name in FEATURE_LABELS:
        return FEATURE_LABELS[feature_name]
    # One hot columns arrive as "task_priority_Critical"
    for raw in CATEGORICAL_FEATURES:
        if feature_name.startswith(raw + "_"):
            return f"{raw.replace('_', ' ').title()} is {feature_name[len(raw) + 1:]}"
    return feature_name.replace("_", " ")


def explain_task(position):
    # position is an integer index into the test split
    row_index = X_test_processed.index[position]
    probability = float(test_probability[position])
    predicted_class = int(probability >= SELECTED_THRESHOLD)

    contributions = pd.Series(shap_values[position], index=FEATURE_NAMES)
    top_drivers = contributions.reindex(contributions.abs().sort_values(ascending=False).index).head(6)

    return {
        "task_id": meta_test.loc[row_index, "task_id"],
        "row_index": row_index,
        "probability": probability,
        "predicted_class": predicted_class,
        "actual_class": int(y_test.loc[row_index]),
        "risk": risk_category(probability),
        "drivers": top_drivers,
        "raw_values": X_test.loc[row_index],
    }

In [ ]:
# Select a genuinely high risk task so the explanation is representative of a flagged case.
high_risk_positions = np.where(test_probability >= 0.60)[0]
example_position = int(high_risk_positions[0]) if len(high_risk_positions) else int(
    np.argmax(test_probability))

example = explain_task(example_position)

summary_frame = pd.DataFrame({
    "field": ["task_id", "predicted probability", "predicted class",
              "risk category", "decision threshold", "actual outcome"],
    "value": [
        example["task_id"],
        f"{example['probability'] * 100:.1f}%",
        example["predicted_class"],
        example["risk"],
        SELECTED_THRESHOLD,
        example["actual_class"],
    ],
})
display(summary_frame)

driver_frame = pd.DataFrame({
    "factor": [readable_label(name) for name in example["drivers"].index],
    "feature": example["drivers"].index,
    "shap_contribution": example["drivers"].values,
    "direction": np.where(example["drivers"].values > 0, "increases risk", "reduces risk"),
})
display(driver_frame.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colors = ["#C44E52" if v > 0 else "#55A868" for v in example["drivers"].values]
ax.barh([readable_label(n) for n in example["drivers"].index],
        example["drivers"].values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.invert_yaxis()
ax.set_title(f"Prediction Drivers for Task {example['task_id']}")
ax.set_xlabel("SHAP contribution to predicted log odds")
plt.tight_layout()
plt.show()

In [ ]:
def business_explanation(item):
    increasing = item["drivers"][item["drivers"] > 0]
    reducing = item["drivers"][item["drivers"] < 0]

    lines = [
        f"Task {item['task_id']}",
        "",
        f"Predicted SLA breach probability: {item['probability'] * 100:.0f}%",
        f"Risk: {item['risk']}",
        "",
        "Main contributing factors:",
    ]
    for rank, name in enumerate(increasing.index[:3], start=1):
        lines.append(f"{rank}. {readable_label(name)}")

    if len(reducing):
        lines.append("")
        lines.append("Offsetting factors:")
        for name in reducing.index[:2]:
            lines.append(f"- {readable_label(name)}")

    action = {
        "High": "Review task assignment and dependency status before the SLA window closes.",
        "Medium": "Add to the manager monitoring list and re-check at the next stand up.",
        "Low": "Continue the normal workflow. No action required.",
    }[item["risk"]]

    lines += ["", "Recommended operational action:", action, "",
              "This is a decision support signal. The manager decides whether to act."]
    return "\n".join(lines)


print(business_explanation(example))

### Interpretation and governance note

The explanation names the conditions that drove the score, not the person assigned to the task.
That distinction is deliberate. `employee_id` was excluded from the feature matrix in Section 6,
so the model cannot attach risk to a named individual, and the factors it surfaces are things a
manager can change: reassignment, dependency escalation, workload rebalancing.

Assignee related features do appear, for example historical SLA rate and average completion
time. These describe fit between a task and an available assignee at a point in time. They must
not be repurposed as a performance rating. A low historical SLA rate can reflect a harder task
mix rather than weaker performance, and the model has no way to distinguish the two. Section 19
sets the operating constraint: the system recommends, the manager decides, and no output of
this model feeds an individual performance process.

## Section 17. Error Analysis

Aggregate metrics say how often the model is wrong. Error analysis says where, which is what
determines whether the failures are tolerable in practice.

In [ ]:
error_frame = X_test.copy()
error_frame["task_id"] = meta_test["task_id"]
error_frame["actual"] = y_test
error_frame["probability"] = test_probability
error_frame["predicted"] = (test_probability >= SELECTED_THRESHOLD).astype(int)

error_frame["outcome"] = np.select(
    [
        (error_frame["actual"] == 1) & (error_frame["predicted"] == 1),
        (error_frame["actual"] == 0) & (error_frame["predicted"] == 0),
        (error_frame["actual"] == 0) & (error_frame["predicted"] == 1),
        (error_frame["actual"] == 1) & (error_frame["predicted"] == 0),
    ],
    ["true positive", "true negative", "false positive", "false negative"],
    default="unclassified",
)

display(error_frame["outcome"].value_counts().to_frame("tasks"))

INSPECTION_COLUMNS = [
    "task_id", "probability", "task_complexity", "current_workload_ratio",
    "sla_buffer_ratio", "dependency_delay_hours", "estimated_vs_sla_ratio",
    "employee_historical_sla_rate",
]

In [ ]:
false_positives = error_frame[error_frame["outcome"] == "false positive"]
false_negatives = error_frame[error_frame["outcome"] == "false negative"]

print("False positives: model predicted breach, task completed within SLA")
print("Most confident examples")
display(false_positives.nlargest(5, "probability")[INSPECTION_COLUMNS].round(4))

print("False negatives: model predicted success, task breached SLA")
print("Least suspected examples")
display(false_negatives.nsmallest(5, "probability")[INSPECTION_COLUMNS].round(4))

In [ ]:
# Compare error groups against correctly classified tasks on the key drivers
profile_columns = [
    "current_workload_ratio", "sla_buffer_ratio", "dependency_delay_hours",
    "estimated_vs_sla_ratio", "task_complexity", "employee_historical_sla_rate",
]

profile = error_frame.groupby("outcome")[profile_columns].mean().round(4)
display(profile)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.histplot(data=error_frame, x="probability", hue="outcome", bins=40,
             element="step", stat="count", common_norm=False, ax=ax)
ax.axvline(SELECTED_THRESHOLD, color="black", linestyle="--", linewidth=1.2,
           label=f"Threshold {SELECTED_THRESHOLD}")
ax.set_title("Predicted Probability Distribution by Outcome Type")
ax.set_xlabel("Predicted breach probability")
ax.set_ylabel("Tasks")
plt.tight_layout()
plt.show()

### Interpretation

Read the profile table before drawing conclusions, since the specific pattern depends on the
run. Two structural observations hold regardless of the exact numbers.

Most errors of both types cluster near the decision threshold rather than at the extremes. The
model is rarely confidently wrong, which is the desirable failure shape: the tasks it
misclassifies are genuinely ambiguous given the information available before completion, not
cases where the signal was clear and the model missed it.

**False positives** are tasks flagged that completed on time. Some of these are not really
errors in operational terms. A task with high workload pressure and a thin SLA buffer was
correctly identified as at risk; it then completed on time because someone absorbed the
pressure. The model cannot observe that recovery, so the outcome is recorded as a mistake even
though the risk assessment was reasonable. The cost is manager review time.

**False negatives** are breaches the model did not flag. These are more expensive
operationally. No signal reached the manager, no intervention was possible, and the SLA was
missed with its full contractual and relationship cost. A false positive costs review capacity
that can be scheduled. A false negative costs an outcome that cannot be recovered once the
window closes, which is the asymmetry that argues for a lower threshold when review capacity
allows it.

One limitation of this analysis should be stated rather than glossed over. The tables show what
distinguishes the error groups in the data. They do not explain why any individual task failed,
because the causes of a specific breach are not recorded in this dataset. Assigning a narrative
cause to a single misclassification here would not be supported by the evidence available.

## Section 18. Model Limitations

These constraints are properties of the current work and should travel with any presentation of
the results.

**The dataset is synthetic.** It was generated from a specified latent process, so the model is
partly recovering relationships that were deliberately built into the data. Performance here
measures whether the pipeline is correctly constructed, not whether the approach works at
ServeNow.

**Synthetic performance does not transfer.** The metrics in Section 13 should not be quoted as
expected production performance. Real operational data contains recording gaps, policy changes
mid history, and drivers absent from this schema entirely.

**Real deployment requires historical task level data.** A meaningful validation needs actual
ServeNow task records with true SLA outcomes, ideally covering at least one full seasonal cycle
so that peak and off peak behaviour are both represented.

**Employee derived features require governance.** Historical SLA rate and average completion
time describe assignee context, but the same fields could be misused as a performance rating.
Access control, a documented purpose limitation, and a review process should be agreed before
these features enter production.

**The output is probabilistic.** A task scored at 70 percent is not a task that will breach. It
belongs to a group in which roughly 70 percent breach, assuming the model is well calibrated.
Calibration was not formally assessed in this notebook and should be checked with a reliability
curve before probabilities are shown to managers as percentages.

**Correlation is not causation.** The model identifies conditions associated with breach. It
does not establish that changing a condition would change the outcome. Reducing workload
because the model ranks workload highly is a reasonable operational hypothesis, and it needs a
controlled test to confirm.

**Drift must be monitored.** Feature distributions and the relationship between features and
outcome will both move as staffing, SLA terms, and customer mix change. Model performance and
input distributions should be tracked on a fixed schedule, with a retraining trigger defined in
advance.

**Feature distributions may shift after deployment.** The model itself changes the process it
predicts. If managers intervene on flagged tasks, those tasks stop breaching, and the model
appears to lose accuracy while actually working as intended. This feedback effect needs an
explicit measurement design, for example holding out a small unflagged control group.

**SLA definitions vary.** Real SLA terms differ by customer contract and task type, and may
change mid contract. A single global model may need to be segmented, or the SLA definition
itself supplied as an explicit input.

## Section 19. Business Translation

### Workflow integration

```
Task created
        |
Task assigned
        |
Model calculates SLA breach probability
        |
        +--- Low risk      -> continue normal workflow
        |
        +--- Medium risk   -> manager monitoring
        |
        +--- High risk     -> recommend intervention
```

### Risk bands and routing

| Band | Probability | Routing | Intent |
|---|---|---|---|
| Low | below 0.30 | Normal workflow | No attention consumed |
| Medium | 0.30 to below 0.60 | Manager monitoring list | Visibility without action |
| High | 0.60 and above | Intervention recommended | Named factors attached |

The bands describe how attention is routed. The decision threshold selected in Section 14
controls which tasks generate an active alert, and the two are set independently: the bands are
a communication device, the threshold is an operational parameter tied to review capacity.

### Potential interventions

- **Reassignment** where the assignee is above capacity and another qualified assignee is not
- **Dependency escalation** where blocking work is the dominant driver
- **Workload balancing** where a whole queue is overloaded rather than one task being unusual
- **Manager review** where the drivers are mixed and judgement is required

The SHAP drivers from Section 16 indicate which of these is relevant for a given task, so the
recommendation is specific rather than a generic warning.

### Operating constraints

**AI recommends. The human manager decides.** No action is taken automatically. The model
produces a ranked list with reasons attached, and a manager exercises judgement on it.

**The model must not automatically penalise employees.** No output of this system feeds an
individual performance process, a rating, or a disciplinary action. The system is designed to
identify work at risk, not to rank people. `employee_id` is excluded from the feature matrix
specifically so the model cannot learn person level risk, and that exclusion should be treated
as a design requirement rather than a modelling convenience.

**A flagged task is a question, not a verdict.** The correct response to a high risk score is a
review of the task's conditions. Interventions such as reassignment carry their own costs and
should not be triggered by a score alone.

## Section 20. Model Artifacts

Five artifacts, sufficient to reproduce inference in a separate process. The native XGBoost JSON
is the primary model artifact because it is version portable, unlike a pickled Python object.

In [ ]:
ARTIFACT_DIR = "servenow_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

MODEL_JSON_PATH = os.path.join(ARTIFACT_DIR, "servenow_sla_breach_xgboost.json")
PREPROCESSOR_PATH = os.path.join(ARTIFACT_DIR, "servenow_preprocessor.joblib")
FEATURE_LIST_PATH = os.path.join(ARTIFACT_DIR, "servenow_feature_list.json")
METADATA_PATH = os.path.join(ARTIFACT_DIR, "servenow_model_metadata.json")
BUNDLE_PATH = os.path.join(ARTIFACT_DIR, "servenow_model_bundle.joblib")
ZIP_PATH = "servenow_sla_breach_model_package.zip"

# 1. Native XGBoost model
final_model.save_model(MODEL_JSON_PATH)

# 2. Preprocessing pipeline
joblib.dump(preprocessor, PREPROCESSOR_PATH)

# 3. Exact feature order expected by the model
with open(FEATURE_LIST_PATH, "w") as handle:
    json.dump({
        "encoded_feature_order": FEATURE_NAMES,
        "raw_categorical_features": CATEGORICAL_FEATURES,
        "raw_numeric_features": NUMERIC_FEATURES,
        "engineered_features": ENGINEERED_FEATURES,
    }, handle, indent=2)

# 4. Metadata, populated from the executed run
METADATA = {
    "model_name": "servenow_sla_breach_xgboost",
    "model_type": "XGBClassifier binary:logistic",
    "target": TARGET,
    "training_date": datetime.now().isoformat(timespec="seconds"),
    "random_seed": RANDOM_SEED,
    "training_rows": int(len(X_train)),
    "validation_rows": int(len(X_validation)),
    "test_rows": int(len(X_test)),
    "feature_count": int(len(FEATURE_NAMES)),
    "selected_threshold": SELECTED_THRESHOLD,
    "test_roc_auc": round(float(test_roc_auc), 6),
    "test_pr_auc": round(float(test_pr_auc), 6),
    "test_precision": round(float(selected_test_metrics["precision"]), 6),
    "test_recall": round(float(selected_test_metrics["recall"]), 6),
    "test_f1": round(float(selected_test_metrics["f1"]), 6),
    "test_accuracy": round(float(selected_test_metrics["accuracy"]), 6),
    "hyperparameters": {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                        for k, v in BEST_PARAMS.items()},
    "xgboost_version": xgb.__version__,
    "sklearn_version": sklearn.__version__,
    "data_source": "synthetic mock dataset",
}

with open(METADATA_PATH, "w") as handle:
    json.dump(METADATA, handle, indent=2)

# 5. Complete inference bundle
joblib.dump({
    "preprocessor": preprocessor,
    "model": final_model,
    "feature_names": FEATURE_NAMES,
    "raw_categorical_features": CATEGORICAL_FEATURES,
    "raw_numeric_features": NUMERIC_FEATURES,
    "threshold": SELECTED_THRESHOLD,
    "metadata": METADATA,
}, BUNDLE_PATH)

for path in [MODEL_JSON_PATH, PREPROCESSOR_PATH, FEATURE_LIST_PATH, METADATA_PATH, BUNDLE_PATH]:
    print(f"{os.path.getsize(path) / 1024:>9,.1f} KB  {path}")

print()
print(json.dumps(METADATA, indent=2))

### Interpretation

The bundle exists for convenience in a single process. The separate artifacts exist because a
production deployment usually needs them apart: the JSON model can be loaded by an XGBoost
runtime in any supported language, whereas the joblib files are tied to the Python object
versions that produced them. The feature list is saved separately because column order is the
most common source of silent inference bugs, where a model runs without error and returns
meaningless probabilities.

Note that `engineer_features` is not serialised. Any inference service must apply the same
function to raw input before calling the preprocessor. In a production build this function
belongs in a shared module imported by both the training job and the serving path, so the two
cannot drift apart.

## Section 21. Reload Test

An artifact that cannot be reloaded to produce identical predictions is not a usable artifact.
This check runs immediately after saving, while the original objects are still in memory.

In [ ]:
# Reload every artifact from disk into fresh objects
reloaded_model = XGBClassifier()
reloaded_model.load_model(MODEL_JSON_PATH)

reloaded_preprocessor = joblib.load(PREPROCESSOR_PATH)

with open(FEATURE_LIST_PATH) as handle:
    reloaded_feature_config = json.load(handle)
reloaded_feature_names = reloaded_feature_config["encoded_feature_order"]

with open(METADATA_PATH) as handle:
    reloaded_metadata = json.load(handle)

reloaded_bundle = joblib.load(BUNDLE_PATH)

print("Model loaded        :", type(reloaded_model).__name__)
print("Preprocessor loaded :", type(reloaded_preprocessor).__name__)
print("Bundle keys         :", sorted(reloaded_bundle.keys()))
print("Threshold in bundle :", reloaded_bundle["threshold"])

In [ ]:
# Feature order must be preserved exactly
assert reloaded_feature_names == FEATURE_NAMES, "Feature order changed after reload"
print("Feature order preserved:", len(reloaded_feature_names), "features")

# Run the full raw to probability path using only reloaded objects
SAMPLE_SIZE = min(200, len(X_test))
sample_raw = X_test.iloc[:SAMPLE_SIZE]

sample_processed = pd.DataFrame(
    reloaded_preprocessor.transform(sample_raw),
    columns=reloaded_feature_names,
    index=sample_raw.index,
).astype(float)

reloaded_probability = reloaded_model.predict_proba(sample_processed)[:, 1]
original_probability = test_probability[:SAMPLE_SIZE]

max_difference = float(np.max(np.abs(reloaded_probability - original_probability)))
print("Sample size            :", SAMPLE_SIZE)
print("Maximum absolute delta :", f"{max_difference:.3e}")

comparison_preview = pd.DataFrame({
    "task_id": meta_test["task_id"].iloc[:SAMPLE_SIZE].values,
    "original": original_probability,
    "reloaded": reloaded_probability,
    "difference": reloaded_probability - original_probability,
}).head(10)
display(comparison_preview.round(8))

In [ ]:
assert np.allclose(reloaded_probability, original_probability, atol=1e-6), (
    "Reloaded predictions do not match the original model. "
    "Check the preprocessor version, the feature order, and the boosting round count."
)

# Verify the bundle path independently of the individual artifacts
bundle_processed = pd.DataFrame(
    reloaded_bundle["preprocessor"].transform(sample_raw),
    columns=reloaded_bundle["feature_names"],
    index=sample_raw.index,
).astype(float)
bundle_probability = reloaded_bundle["model"].predict_proba(bundle_processed)[:, 1]

assert np.allclose(bundle_probability, original_probability, atol=1e-6), (
    "Bundle predictions do not match the original model."
)

print("Model reload test passed.")

### Interpretation

The check runs the complete raw input to probability path using only objects read back from
disk, which is the path a production service would follow. Verifying the model alone would miss
the two most common deployment failures: a preprocessor that encodes categories in a different
order, and a boosting round count that differs between the saved and in memory models.

The second failure is why Section 12 refits the final model without early stopping at the
discovered round count. A model trained with early stopping carries a `best_iteration`
attribute that some XGBoost versions apply at prediction time and others do not preserve
through serialisation, which produces a small and easily missed probability shift after reload.
Removing the dependency entirely is more reliable than handling every version case.

## Section 22. Google Colab Download

All artifacts are packaged into a single ZIP, with individual downloads also available.

In [ ]:
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in [MODEL_JSON_PATH, PREPROCESSOR_PATH, FEATURE_LIST_PATH,
                 METADATA_PATH, BUNDLE_PATH]:
        archive.write(path, arcname=os.path.basename(path))

print("Package created:", ZIP_PATH)
print(f"Size: {os.path.getsize(ZIP_PATH) / 1024:,.1f} KB")
print()
with zipfile.ZipFile(ZIP_PATH) as archive:
    for info in archive.infolist():
        print(f"  {info.file_size / 1024:>9,.1f} KB  {info.filename}")

In [ ]:
# Download from Colab. Skipped automatically outside Colab.
try:
    from google.colab import files

    files.download(ZIP_PATH)

    # Uncomment to download artifacts individually instead of the package.
    # files.download(MODEL_JSON_PATH)
    # files.download(PREPROCESSOR_PATH)
    # files.download(FEATURE_LIST_PATH)
    # files.download(METADATA_PATH)
    # files.download(BUNDLE_PATH)

except ImportError:
    print("Not running in Colab. Artifacts are on disk at:")
    print(" ", os.path.abspath(ARTIFACT_DIR))
    print(" ", os.path.abspath(ZIP_PATH))

## Section 23. Final Model Summary

The card below is printed from the executed run, so every figure it reports comes from this
notebook rather than from a written estimate.

In [ ]:
def render_model_card():
    top_ten = global_importance.head(10)

    lines = []
    lines.append("=" * 68)
    lines.append("SERVENOW SLA BREACH PREDICTION, MODEL CARD")
    lines.append("=" * 68)
    lines.append("")
    lines.append("MODEL")
    lines.append("  XGBoost Binary Classifier")
    lines.append(f"  xgboost {xgb.__version__}, scikit-learn {sklearn.__version__}")
    lines.append(f"  Trained {METADATA['training_date']}, seed {RANDOM_SEED}")
    lines.append("")
    lines.append("OBJECTIVE")
    lines.append("  Predict the probability of SLA breach before task completion.")
    lines.append("")
    lines.append("TARGET")
    lines.append(f"  {TARGET}  (1 = breached, 0 = within SLA)")
    lines.append("")
    lines.append("DATA")
    lines.append(f"  Train {len(X_train):,} rows | Validation {len(X_validation):,} rows | "
                 f"Test {len(X_test):,} rows")
    lines.append(f"  Encoded features: {len(FEATURE_NAMES)}")
    lines.append(f"  Positive class rate: {y_all.mean():.4f}")
    lines.append("")
    lines.append("KEY FEATURES, top 10 by mean absolute SHAP")
    for rank, (name, value) in enumerate(top_ten.items(), start=1):
        lines.append(f"  {rank:>2}. {name:<38} {value:.5f}")
    lines.append("")
    lines.append("TEST PERFORMANCE")
    lines.append(f"  ROC-AUC   : {test_roc_auc:.4f}")
    lines.append(f"  PR-AUC    : {test_pr_auc:.4f}   (baseline {y_test.mean():.4f})")
    lines.append(f"  Precision : {selected_test_metrics['precision']:.4f}")
    lines.append(f"  Recall    : {selected_test_metrics['recall']:.4f}")
    lines.append(f"  F1        : {selected_test_metrics['f1']:.4f}")
    lines.append(f"  Accuracy  : {selected_test_metrics['accuracy']:.4f}")
    lines.append("")
    lines.append("DECISION THRESHOLD")
    lines.append(f"  Selected: {SELECTED_THRESHOLD}")
    lines.append("  Basis   : highest F1 on the validation set, chosen without reference")
    lines.append("            to the test set. Equal weighting of precision and recall is a")
    lines.append("            placeholder for real breach and review costs, which should")
    lines.append("            replace it before deployment.")
    lines.append(f"  At this threshold {selected_test_metrics['flagged_pct']:.1f}% of test tasks")
    lines.append("  are routed for review.")
    lines.append("")
    lines.append("BUSINESS USE")
    lines.append("  Prioritise tasks requiring management intervention. The model produces a")
    lines.append("  ranked risk signal with named contributing factors. AI recommends, the")
    lines.append("  manager decides. No output feeds an individual performance process.")
    lines.append("")
    lines.append("LIMITATION")
    lines.append("  The training data is synthetic. These metrics demonstrate that the")
    lines.append("  pipeline is correctly constructed. They do not establish production")
    lines.append("  effectiveness and should not be quoted as expected live performance.")
    lines.append("")
    lines.append("NEXT STEP")
    lines.append("  Train and validate this same pipeline on real historical ServeNow task")
    lines.append("  data covering at least one full seasonal cycle. Recompute all employee")
    lines.append("  and similar task aggregates strictly as of each task's creation time,")
    lines.append("  assess probability calibration, and agree a drift monitoring schedule")
    lines.append("  before any operational use.")
    lines.append("")
    lines.append("=" * 68)
    return "\n".join(lines)


print(render_model_card())

In [ ]:
# Persist the model card alongside the artifacts
CARD_PATH = os.path.join(ARTIFACT_DIR, "servenow_model_card.txt")
with open(CARD_PATH, "w") as handle:
    handle.write(render_model_card())

with zipfile.ZipFile(ZIP_PATH, "a", zipfile.ZIP_DEFLATED) as archive:
    archive.write(CARD_PATH, arcname=os.path.basename(CARD_PATH))

print("Model card saved to:", CARD_PATH)
print("Notebook complete.")

### Closing note

The notebook runs end to end and produces a verified, reloadable model package. What it does
not do is establish that the approach works at ServeNow, because the training data is synthetic
and the model is partly recovering relationships that were designed into it.

The single most important step before any operational use is to rebuild the three historical
aggregate features on real data with strict as of timestamps. `employee_historical_sla_rate`,
`employee_avg_completion_hours`, and `similar_task_avg_hours` are all computed from prior tasks,
and computing them over a full history that includes later closures is look ahead leakage. It
produces a column that looks entirely ordinary, passes every automated check in Section 6, and
inflates offline performance in a way that only becomes visible after deployment.